In [0]:
from pyspark.sql.functions import sum as spark_sum, col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df_journey = spark.table("mini_cap.default.booking_master")

airline_revenue_report = (
    df_journey.groupBy("airline")
    .agg(spark_sum("ticket_price").alias("revenue"))
    .orderBy(col("revenue").desc())
)
print("=== Airline Revenue Report ===")
airline_revenue_report.show()

route_performance_report = (
    df_journey.groupBy("from_city", "to_city")
    .agg(spark_sum("ticket_price").alias("revenue"))
    .orderBy(col("revenue").desc())
)
print("=== Route Performance Report ===")
route_performance_report.show()

passenger_preference_report = df_journey.select("passenger_name", "meal", "seat").distinct()
print("=== Passenger Preference Report ===")
passenger_preference_report.show()

flight_delay_report = df_journey.select("flight_id", "status").distinct()
print("=== Flight Delay Report ===")
flight_delay_report.show()

flight_revenue = (
    df_journey.groupBy("flight_id")
    .agg(spark_sum("ticket_price").alias("revenue"))
)
window_spec = Window.orderBy(col("revenue").desc())
top_revenue_flights_report = (
    flight_revenue
    .withColumn("rank", row_number().over(window_spec))
    .orderBy("rank")
)
print("=== Top Revenue Flights Report ===")
top_revenue_flights_report.show()

print("generate_reports complete.")

=== Airline Revenue Report ===
+---------+-------+
|  airline|revenue|
+---------+-------+
|   Indigo|  90000|
|  Vistara|  71500|
|Air India|  68000|
|    Akasa|  62000|
+---------+-------+

=== Route Performance Report ===
+---------+---------+-------+
|from_city|  to_city|revenue|
+---------+---------+-------+
|Hyderabad|    Delhi|  39000|
|Bangalore|Hyderabad|  38000|
|    Delhi|  Chennai|  28000|
|Hyderabad|  Kolkata|  26500|
|  Chennai|Bangalore|  25000|
|  Chennai|     Pune|  24000|
|Bangalore|   Mumbai|  23500|
|    Delhi|Hyderabad|  18000|
|Hyderabad|      Goa|  16000|
|  Kolkata|Bangalore|  10500|
|     Pune|    Delhi|  10000|
|   Mumbai|Hyderabad|   9500|
|   Mumbai|  Chennai|   9000|
|    Delhi|   Mumbai|   7500|
|      Goa|    Delhi|   7000|
+---------+---------+-------+

=== Passenger Preference Report ===
+--------------+-------+------+
|passenger_name|   meal|  seat|
+--------------+-------+------+
|  Rahul Sharma|    Veg|Window|
|   Priya Reddy|Non-Veg| Aisle|
|    Ami

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------+----+
|flight_id|revenue|rank|
+---------+-------+----+
|     F101|  39000|   1|
|     F103|  38000|   2|
|     F109|  28000|   3|
|     F107|  26500|   4|
|     F105|  25000|   5|
|     F113|  24000|   6|
|     F110|  23500|   7|
|     F115|  18000|   8|
|     F111|  16000|   9|
|     F114|  10500|  10|
|     F106|  10000|  11|
|     F108|   9500|  12|
|     F102|   9000|  13|
|     F104|   7500|  14|
|     F112|   7000|  15|
+---------+-------+----+

generate_reports complete.
